# Deep Learning Recommenders (Part 1)

## MovieLens 100K data
In order to make direct comparison with classical methods introduced in Week 3, we will use the same MovieLens 100K data for now.

In [ ]:
# Before we start, let's first download the data and unzip it. The data is available at: https://files.grouplens.org/datasets/movielens/ml-100k.zip
# Please skip this step if you already have the data from the previous lectures.


import pandas as pd
import zipfile
import os
import urllib.request

# Step 1: Download the dataset
url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
zip_path = "ml-100k.zip"
data_dir = "ml-100k"

if not os.path.exists(zip_path):
    print("Downloading MovieLens 100k dataset...")
    urllib.request.urlretrieve(url, zip_path)
    print("Download complete.")

# Step 2: Unzip the dataset
if not os.path.exists(data_dir):
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall()
    print("Extraction complete.")


Download complete.
Extracting dataset...
Extraction complete.


In [3]:
# Step 3: Load ratings data
ratings = pd.read_csv(
    os.path.join(data_dir, 'u.data'),
    sep='\t',
    names=['user_id', 'item_id', 'rating', 'timestamp'],
    encoding='latin-1'
)

# Preview
print("Ratings sample:")
print(ratings.head())

# Check the shape of the datasets
print("Ratings shape:", ratings.shape) #100k ratings
# Print the number of unique users and movies
print("Number of unique users:", ratings['user_id'].nunique())
print("Number of unique movies:", ratings['item_id'].nunique())


Ratings sample:
   user_id  item_id  rating  timestamp
0      196      242       3  881250949
1      186      302       3  891717742
2       22      377       1  878887116
3      244       51       2  880606923
4      166      346       1  886397596
Ratings shape: (100000, 4)
Number of unique users: 943
Number of unique movies: 1682


## Train-Test Split

In [4]:
from sklearn.model_selection import train_test_split

# Split the ratings into train and test sets (80/20 split)
# Note: We are using a random split here.
# But in practice, we could split based on time, which would be more realistic for a recommendation system (i.e., we would be using historical data to predict future ratings).
train_df, test_df = train_test_split(ratings, test_size=0.2, random_state=66)

## Neural Collaborative Filtering

The code for neural collaborative filtering can be mostly inherited from the matrix factorization's code. The main change is to replace the inner product of user- and item-embeddings by concatenation, followed by a few fully-connected layers to generate the output.

### Data Preparation

In [ ]:
# From this part, we will shift to Tensorflow/Keras. 
# Although matrix factorization can also be done with numpy, using Keras allows a smooth transition to deep learning models, which we will cover in the next few lectures.

import numpy as np
num_users = ratings['user_id'].max() #Obtain the largest possible number of unique users
num_items = ratings['item_id'].max() #Obtain the largest possible number of unique items
d = 30 #Number of latent factors
## you can use whatever number you want -- how to optimize this? 

print(f"Number of users: {num_users}")
print(f"Number of items: {num_items}")

Number of users: 943
Number of items: 1682


### Model Building

In [7]:
import tensorflow as tf

from keras.layers import Input, Embedding, Flatten, Dense, Concatenate
from keras.models import Model
from keras import backend as K

# Clear any previous session to avoid clutter
K.clear_session()

# creating user embedding path
user_input = Input(shape=[1], name="User-Input")
user_embedding = Embedding(num_users+1, d)(user_input) #Use num_users+1 to account for 0-indexing
user_embedding = Flatten()(user_embedding)

# creating item embedding path
item_input = Input(shape=[1], name="Item-Input")
item_embedding = Embedding(num_items+1, d)(item_input)
item_embedding = Flatten()(item_embedding)

# creating concatenated path
concat = Concatenate()([user_embedding, item_embedding])

# use a few dense layers to learn the interaction
x = Dense(32, activation='relu')(concat)
x = Dense(16, activation='relu')(x)
out = Dense(1)(x)  # No activation for regression

# creating model
model = Model([user_input, item_input], out)

model.compile(loss='mean_squared_error', optimizer=tf.keras.optimizers.Adam(), metrics=['mae'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ User-Input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Item-Input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 30)     │     28,320 │ User-Input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 30)     │     50,490 │ Item-Input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 30)        │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 30)        │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 60)        │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      1,952 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │        528 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │         17 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 81,307 (317.61 KB)

 Trainable params: 81,307 (317.61 KB)

 Non-trainable params: 0 (0.00 B)

### Training

In [8]:
# Early stopping, which allows us to stop training when the validation loss stops improving (optional)
# We will monitor the validation loss and stop training if it doesn't improve for 20 epochs.
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=20, restore_best_weights=True
)
# Training
# Input data for the model is a list of two arrays: user IDs and item IDs.
# The output is the rating.
model.fit([train_df.user_id, train_df.item_id], train_df.rating, epochs=100, validation_split=0.15, callbacks=[early_stopping])

Epoch 1/100
2125/2125 ━━━━━━━━━━━━━━━━━━━━ 1s 436us/step - loss: 1.3371 - mae: 0.8722 - val_loss: 0.9247 - val_mae: 0.7600
Epoch 2/100
2125/2125 ━━━━━━━━━━━━━━━━━━━━ 1s 405us/step - loss: 0.8920 - mae: 0.7478 - val_loss: 0.9086 - val_mae: 0.7423
Epoch 3/100
2125/2125 ━━━━━━━━━━━━━━━━━━━━ 1s 400us/step - loss: 0.8553 - mae: 0.7303 - val_loss: 0.8864 - val_mae: 0.7441
Epoch 4/100
2125/2125 ━━━━━━━━━━━━━━━━━━━━ 1s 407us/step - loss: 0.8237 - mae: 0.7151 - val_loss: 0.8801 - val_mae: 0.7426
Epoch 5/100
2125/2125 ━━━━━━━━━━━━━━━━━━━━ 1s 421us/step - loss: 0.7936 - mae: 0.7002 - val_loss: 0.8786 - val_mae: 0.7338
Epoch 6/100
2125/2125 ━━━━━━━━━━━━━━━━━━━━ 1s 409us/step - loss: 0.7628 - mae: 0.6858 - val_loss: 0.8856 - val_mae: 0.7421
Epoch 7/100
2125/2125 ━━━━━━━━━━━━━━━━━━━━ 1s 404us/step - loss: 0.7321 - mae: 0.6711 - val_loss: 0.8949 - val_mae: 0.7415
Epoch 8/100
2125/2125 ━━━━━━━━━━━━━━━━━━━━ 1s 418us/step - loss: 0.7043 - mae: 0.6574 - val_loss: 0.8974 - val_mae: 0.7431
Epoch 9/100
2125

### Evaluation

In [10]:
# Evaluation
mse, mae = model.evaluate([test_df.user_id, test_df.item_id], test_df.rating)
rmse=np.sqrt(mse)
print(f"RMSE_NCF: {rmse:.4f}") #0.9319
print(f"MAE_NCF: {mae:.4f}")   #0.7304
# The RMSE and MAE are similar to those of matrix factorization. 
# RMSE_MF: 0.9281, MAE_MF: 0.7297

  1/625 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.6639 - mae: 0.6301

625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 327us/step - loss: 0.8620 - mae: 0.7263
RMSE_NCF: 0.9285
MAE_NCF: 0.7263


## Wide \& Deep


### Load libraries and datasets

In [11]:
# Load all necessary libraries
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Concatenate, Dense, Add
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError

# Load datasets
# Note that, Wide & Deep is able to utilize features in addition to the utility matrix.
# Therefore, we load not only the ratings, but also the users and items datasets.
ratings = pd.read_csv('https://files.grouplens.org/datasets/movielens/ml-100k/u.data', 
                      sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])
users = pd.read_csv('https://files.grouplens.org/datasets/movielens/ml-100k/u.user',
                    sep='|', names=['user_id', 'age', 'gender', 'occupation', 'zip_code'])
items = pd.read_csv('https://files.grouplens.org/datasets/movielens/ml-100k/u.item',
                    sep='|', encoding='latin-1', header=None,
                    names=['item_id', 'title', 'release_date', 'video_release_date', 'IMDb_URL'] + [f'genre_{i}' for i in range(19)])

# Check how each dataset looks like
ratings.head(), users.head(), items.head()

(   user_id  item_id  rating  timestamp
 0      196      242       3  881250949
 1      186      302       3  891717742
 2       22      377       1  878887116
 3      244       51       2  880606923
 4      166      346       1  886397596,
    user_id  age gender  occupation zip_code
 0        1   24      M  technician    85711
 1        2   53      F       other    94043
 2        3   23      M      writer    32067
 3        4   24      M  technician    43537
 4        5   33      F       other    15213,
    item_id              title release_date  video_release_date  \
 0        1   Toy Story (1995)  01-Jan-1995                 NaN   
 1        2   GoldenEye (1995)  01-Jan-1995                 NaN   
 2        3  Four Rooms (1995)  01-Jan-1995                 NaN   
 3        4  Get Shorty (1995)  01-Jan-1995                 NaN   
 4        5     Copycat (1995)  01-Jan-1995                 NaN   
 
                                             IMDb_URL  genre_0  genre_1  \
 0  http:

### Data Processing (including the features for the "wide" part)

In [12]:
# Data Preprocessing
# More operations are needed to preprocess the data because of the additional features.

# Preprocess user features
# For simplicity, we categorize the age into bins and one-hot encode it.
user_df = users[['user_id', 'age', 'gender', 'occupation']].copy()
user_df['age'] = pd.cut(user_df['age'], bins=[0, 18, 25, 35, 45, 50, 56, 100], labels=False)
user_enc = pd.get_dummies(user_df[['age', 'gender', 'occupation']].astype(str))
user_df = pd.concat([user_df[['user_id']], user_enc], axis=1)

# Preprocess item features
# We will also one-hot encode the genres and release year.
items['release_year'] = pd.to_datetime(items['release_date'], errors='coerce').dt.year.fillna(0).astype(int)
items['release_bucket'] = pd.cut(items['release_year'], bins=[0, 1980, 1985, 1990, 1995, 2000], labels=False)
item_enc = pd.get_dummies(items[[f'genre_{i}' for i in range(19)]].astype(str))
item_release = pd.get_dummies(items['release_bucket'].astype(str), prefix='release')
item_df = pd.concat([items[['item_id']], item_release, item_enc], axis=1)

# Merge with ratings
# Merge the ratings with the user and item features
ratings = ratings.merge(user_df, on='user_id')
ratings = ratings.merge(item_df, on='item_id')

# Check the number of unique users and items
n_users = len(ratings['user_id'].unique())
n_items = len(ratings['item_id'].unique())

# Check how the ratings dataset looks like after merging with user and item features
print(ratings.head())
print(f"Number of users: {n_users}")
print(f"Number of items: {n_items}")


   user_id  item_id  rating  timestamp  age_0  age_1  age_2  age_3  age_4  \
0      196      242       3  881250949  False  False  False  False   True   
1      186      302       3  891717742  False  False  False   True  False   
2       22      377       1  878887116  False   True  False  False  False   
3      244       51       2  880606923  False  False   True  False  False   
4      166      346       1  886397596  False  False  False  False   True   

   age_5  ...  genre_14_0  genre_14_1  genre_15_0  genre_15_1  genre_16_0  \
0  False  ...        True       False        True       False        True   
1  False  ...        True       False        True       False       False   
2  False  ...        True       False        True       False        True   
3  False  ...       False        True        True       False        True   
4  False  ...        True       False        True       False        True   

   genre_16_1  genre_17_0  genre_17_1  genre_18_0  genre_18_1  
0       Fa

### Train-Test Split

In [13]:
# Train-test split
train, test = train_test_split(ratings, test_size=0.2, random_state=42)

# Prepare inputs
# This includes the wide part and the deep part.
# The wide part will include the user and item features.
# The deep part will include the user and item IDs.
X_train_wide_user = train[user_enc.columns].values
X_train_wide_item = train[item_release.columns.tolist() + item_enc.columns.tolist()].values
X_train_deep = [train['user_id'].values, train['item_id'].values]
y_train = train['rating'].values

# Prepare test data
# The same preprocessing is applied to the test data.
X_test_wide_user = test[user_enc.columns].values
X_test_wide_item = test[item_release.columns.tolist() + item_enc.columns.tolist()].values
X_test_deep = [test['user_id'].values, test['item_id'].values]
y_test = test['rating'].values

# print the head of X_train_wide_user and X_train_wide_item and X_train_deep
print(X_train_wide_user[:5])
print(X_train_wide_item[:5])
print(X_train_deep[0][:5])
print(X_train_deep[1][:5])


[[False False False  True False False False  True False False False False
  False False False False  True False False False False False False False
  False False False False False False]
 [False False False False False  True False False  True False False False
  False False False  True False False False False False False False False
  False False False False False False]
 [False False False False  True False False  True False False False False
  False False False False  True False False False False False False False
  False False False False False False]
 [False  True False False False False False False  True False False False
  False False False False False False False False False False False False
  False False False  True False False]
 [ True False False False False False False False  True False False False
  False False False False False False False False False False False False
  False False False  True False False]]
[[ True False False False False  True False  True False False  T

### Model Construction

In [14]:
# Model Building
from keras import backend as K
K.clear_session()

# Deep part
embedding_size = 32
user_input = Input(shape=(1,), name='user_input')
item_input = Input(shape=(1,), name='item_input')

user_embedding = Embedding(n_users+1, embedding_size)(user_input)
item_embedding = Embedding(n_items+1, embedding_size)(item_input)
user_vec = Flatten()(user_embedding)
item_vec = Flatten()(item_embedding)
deep_input = Concatenate()([user_vec, item_vec])
deep = Dense(32, activation='relu')(deep_input)
deep = Dense(16, activation='relu')(deep)

# Wide part
wide_user_input = Input(shape=(X_train_wide_user.shape[1],), name='wide_user_input') #The size of the input is the number of features, which can be different across different projects
wide_item_input = Input(shape=(X_train_wide_item.shape[1],), name='wide_item_input')
wide = Concatenate()([wide_user_input, wide_item_input])

# Combine
combined = Concatenate()([deep, wide])
output = Dense(1)(combined)

# Model
model = Model(inputs=[user_input, item_input, wide_user_input, wide_item_input], outputs=output)
model.compile(optimizer=Adam(0.001), loss=MeanSquaredError(), metrics=[RootMeanSquaredError()])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 32)     │     30,208 │ user_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 32)     │     53,856 │ item_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 32)        │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 32)        │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 64)        │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      2,080 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_user_input     │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_item_input     │ (None, 43)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │        528 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 73)        │          0 │ wide_user_input[… │
│ (Concatenate)       │                   │            │ wide_item_input[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 89)        │          0 │ dense_1[0][0],    │
│ (Concatenate)       │                   │            │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │         90 │ concatenate_2[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 86,762 (338.91 KB)

 Trainable params: 86,762 (338.91 KB)

 Non-trainable params: 0 (0.00 B)

### Model Training

In [15]:
# Early stopping, which allows us to stop training when the validation loss stops improving (optional)
# We will monitor the validation loss and stop training if it doesn't improve for 20 epochs.
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=20, restore_best_weights=True
)
# Training
# Input data for the model is a list of two arrays: user IDs and item IDs.
# The output is the rating.
model.fit(x=[X_train_deep[0], X_train_deep[1], X_train_wide_user, X_train_wide_item],
          y=y_train, epochs=100, validation_split=0.15, callbacks=[early_stopping])

Epoch 1/100
2125/2125 ━━━━━━━━━━━━━━━━━━━━ 2s 500us/step - loss: 1.2291 - root_mean_squared_error: 1.1087 - val_loss: 0.9202 - val_root_mean_squared_error: 0.9593
Epoch 2/100
2125/2125 ━━━━━━━━━━━━━━━━━━━━ 1s 458us/step - loss: 0.8815 - root_mean_squared_error: 0.9389 - val_loss: 0.8894 - val_root_mean_squared_error: 0.9431
Epoch 3/100
2125/2125 ━━━━━━━━━━━━━━━━━━━━ 1s 456us/step - loss: 0.8493 - root_mean_squared_error: 0.9216 - val_loss: 0.8755 - val_root_mean_squared_error: 0.9357
Epoch 4/100
2125/2125 ━━━━━━━━━━━━━━━━━━━━ 1s 450us/step - loss: 0.8230 - root_mean_squared_error: 0.9072 - val_loss: 0.8811 - val_root_mean_squared_error: 0.9387
Epoch 5/100
2125/2125 ━━━━━━━━━━━━━━━━━━━━ 1s 463us/step - loss: 0.7948 - root_mean_squared_error: 0.8915 - val_loss: 0.8679 - val_root_mean_squared_error: 0.9316
Epoch 6/100
2125/2125 ━━━━━━━━━━━━━━━━━━━━ 1s 462us/step - loss: 0.7664 - root_mean_squared_error: 0.8754 - val_loss: 0.8649 - val_root_mean_squared_error: 0.9300
Epoch 7/100
2125/2125 

### Model Evaluation

In [16]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Predict on test set
y_pred = model.predict([
    X_test_deep[0],   # user IDs
    X_test_deep[1],   # item IDs
    X_test_wide_user, # user wide features
    X_test_wide_item  # item wide features
]).flatten()

# Calculate RMSE and MAE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

# Print results
print(f"Test RMSE: {rmse:.4f}") #0.9265
print(f"Test MAE:  {mae:.4f}")  #0.7256
#The results are slightly improved than NCF due to the use of additional information (i.e., user and movie features).

625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 238us/step
Test RMSE: 0.9347
Test MAE:  0.7304
